# 03 – EDA và trực quan hóa IMDb

**Đầu vào:** `data/processed/imdb_cleaned.csv`.

**Đầu ra:** Các biểu đồ PNG trong `report/figures/` và bảng thống kê trong `data/processed/`.

Không kết luận quan hệ nhân quả từ phân tích tương quan.

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()

if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent

DATA = ROOT / 'data' / 'processed'
PLOTS = ROOT / 'report' / 'figures'

PLOTS.mkdir(parents=True, exist_ok=True)

path = DATA / 'imdb_cleaned.csv'

if not path.exists():
    raise FileNotFoundError("Không tìm thấy imdb_cleaned.csv")


In [ ]:

# Kiểm tra dung lượng file
size_mb = path.stat().st_size / (1024 ** 2)

print(f"Dung lượng CSV: {size_mb:.2f} MB", flush=True)

# Đọc dữ liệu
print("Đang đọc CSV...", flush=True)

start = time.perf_counter()

df = pd.read_csv(path)

print(f"Đọc xong sau: {time.perf_counter() - start:.2f} giây", flush=True)

# Kiểm tra dữ liệu
print("Kích thước:", df.shape)
print(f"RAM DataFrame: {df.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB")

display(df.head())

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", 50)

# Hàm lưu biểu đồ
def saveplot(filename):
    plt.tight_layout()
    plt.savefig(PLOTS / filename, dpi=120)
    plt.close()

## 3.1. Thống kê mô tả, giá trị thiếu

In [ ]:
features_numeric = ['imdb_rating', 'year', 'duration_minutes', 'metascore']
summary = df[features_numeric].describe().T
summary['missing'] = df[features_numeric].isna().sum()
summary['missing_%'] = (df[features_numeric].isna().mean() * 100).round(2)
display(summary)
summary.to_csv(DATA / 'eda_numeric_summary.csv', encoding='utf-8-sig')

display(df[['certificates','director','genres']].describe().T)

In [ ]:
# Biểu đồ số liệu thiếu TRƯỚC khi điền 'Unknown' (nếu có file).
missing_path = DATA / 'missing_before_fill.csv'
if missing_path.exists():
    missing_df = pd.read_csv(missing_path)
else:
    missing_df = pd.DataFrame({'column':df.columns,
                               'missing_percent':df.isna().mean().values * 100})
shown = missing_df.nlargest(12, 'missing_percent').sort_values('missing_percent')
fig, ax = plt.subplots(figsize=(9,5))
sns.barplot(data=shown, x='missing_percent', y='column', ax=ax, color='steelblue')
ax.set(xlabel='Tỷ lệ thiếu (%)', ylabel='Thuộc tính', title='Tỷ lệ dữ liệu thiếu theo cột')
saveplot('01_missing_values.png')

## 3.2. Phân phối thuộc tính số: IMDb Rating, MetaScore và thời lượng

In [ ]:
for col, title, fn, bins in [
    ('imdb_rating', 'Phân phối điểm IMDb', '02_imdb_rating_hist.png', 20),
    ('metascore', 'Phân phối MetaScore', '03_metascore_hist.png', 20),
    ('duration_minutes', 'Phân phối thời lượng phim', '04_duration_hist.png', 30),
]:
    values = pd.to_numeric(df[col], errors='coerce').dropna()
    if values.empty:
        print('Bỏ qua biểu đồ', col, 'vì không có số liệu.')
        continue
    fig, ax = plt.subplots(figsize=(9,5))
    sns.histplot(values, bins=bins, kde=(len(values) >= 5), ax=ax, color='steelblue')
    ax.set(title=title, xlabel=col, ylabel='Số phim')
    saveplot(fn)

## 3.3. Số phim theo thập niên và theo thể loại

In [ ]:
years = pd.to_numeric(df['year'], errors='coerce').dropna()
if not years.empty:
    decades = (years.floordiv(10) * 10).astype(int).value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(10,5))
    sns.barplot(x=decades.index.astype(str), y=decades.values, ax=ax, color='steelblue')
    ax.tick_params(axis='x', labelrotation=45)
    ax.set(title='Số phim theo thập niên', xlabel='Thập niên', ylabel='Số phim')
    saveplot('05_films_by_decade.png')

all_genres = (df['genres'].dropna().astype('string').str.split(',')
              .explode().str.strip())
all_genres = all_genres[all_genres.ne('Unknown') & all_genres.ne('')]
genre_counts = all_genres.value_counts().head(12)
if not genre_counts.empty:
    fig, ax = plt.subplots(figsize=(10,6))
    sns.barplot(x=genre_counts.values, y=genre_counts.index, ax=ax, color='steelblue')
    ax.set(title='Top 12 thể loại xuất hiện nhiều nhất', xlabel='Số phim', ylabel='Thể loại')
    saveplot('06_top_genres.png')
display(genre_counts.rename('so_phim').to_frame())

## 3.4. IQR và biểu đồ hộp

Ngoại lệ theo IQR là điểm nằm ngoài [Q1−1.5×IQR, Q3+1.5×IQR]. Không xóa dữ liệu chỉ vì nằm ngoài ngưỡng.

In [ ]:
def get_iqr(frame, columns):
    result = []
    for c in columns:
        s = pd.to_numeric(frame[c], errors='coerce').dropna()
        if s.empty:
            continue
        q1, q3 = s.quantile([.25,.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
        result.append({'column':c,'Q1':q1,'Q3':q3,'IQR':iqr,
                       'lower':lo,'upper':hi,
                       'outlier_count':int(((s<lo)|(s>hi)).sum())})
    return pd.DataFrame(result)

iqr = get_iqr(df, ['imdb_rating', 'duration_minutes', 'metascore', 'year'])
display(iqr)

for c, filename in [('duration_minutes','07_duration_boxplot.png'),
                     ('imdb_rating','08_imdb_rating_boxplot.png')]:
    s = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(s) > 0:
        fig, ax = plt.subplots(figsize=(9,3))
        sns.boxplot(x=s, ax=ax, color='steelblue')
        ax.set(title='Boxplot: ' + c, xlabel=c)
        saveplot(filename)

## 3.5. Tương quan giữa các biến số

In [ ]:
num = df[features_numeric].apply(pd.to_numeric, errors='coerce')
correlation = num.corr(method='pearson')
display(correlation)
if not correlation.isna().all().all():
    fig, ax = plt.subplots(figsize=(7,5))
    sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
    ax.set(title='Ma trận tương quan Pearson')
    saveplot('09_correlation_heatmap.png')

pair = df[['metascore','imdb_rating']].dropna()
if len(pair) >= 2:
    fig, ax = plt.subplots(figsize=(9,5))
    sns.scatterplot(data=pair, x='metascore', y='imdb_rating', alpha=.5, ax=ax, color='steelblue')
    ax.set(title='MetaScore và IMDb Rating', xlabel='MetaScore', ylabel='IMDb Rating')
    saveplot('10_rating_vs_metascore.png')
    print('Pearson r:', round(pair['metascore'].corr(pair['imdb_rating']), 3))

## 3.6. So sánh IMDb Rating theo thể loại (lọc nhóm có đủ mẫu)

In [ ]:
movies_genres = df.loc[df['imdb_rating'].notna() & df['genres'].ne('Unknown'),
                        ['genres','imdb_rating']].copy()
movies_genres['genre'] = movies_genres['genres'].str.split(',')
movies_genres = movies_genres.explode('genre')
movies_genres['genre'] = movies_genres['genre'].str.strip()

genre_stats = (movies_genres.groupby('genre')['imdb_rating']
               .agg(so_phim='count', diem_trung_binh='mean', trung_vi='median')
               .sort_values('so_phim', ascending=False))
# Lấy từ 10 phim trở lên, tránh suy diễn từ những nhóm cực nhỏ.
valid_genres = genre_stats.loc[genre_stats['so_phim'] >= 10].head(12)
display(valid_genres)
if not valid_genres.empty:
    fig, ax = plt.subplots(figsize=(10,6))
    sns.barplot(x=valid_genres['diem_trung_binh'], y=valid_genres.index, ax=ax, color='steelblue')
    ax.set(title='Điểm IMDb trung bình theo thể loại (ít nhất 10 phim)',
           xlabel='Điểm trung bình', ylabel='Thể loại')
    saveplot('11_mean_rating_by_genre.png')

genre_stats.to_csv(DATA / 'eda_genre_summary.csv', encoding='utf-8-sig')
print('Đã lưu các biểu đồ tại:', PLOTS)